# Grab-and-Go: YOLO11n Training on RPC Dataset

Trains a YOLO11n model on the filtered RPC retail dataset (76 classes).

## Pre-work
1. Create the dataset package:
   - Light: `python scripts/create_lightweight.py --samples-per-class 50`
   - Then: `python scripts/package_for_colab.py --source data_light --out data_light.zip`
2. Upload `data_light.zip` (or `grabandgo_dataset.zip` for full) to Google Drive
3. Set the ZIP_FILENAME in the next cell

In [ ]:
# --- CONFIG ---
# Set this to your zip file name (uploaded to Drive root folder)
ZIP_FILENAME = "data_light.zip"  # or "grabandgo_dataset.zip" for full
# --- END CONFIG ---

import os
import zipfile
from pathlib import Path

ZIP_SOURCE = f"/content/drive/MyDrive/{ZIP_FILENAME}"
EXTRACT_DIR = Path("/content/grabandgo")

# Mount Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Extract dataset
if not EXTRACT_DIR.exists():
    print(f"Extracting {ZIP_SOURCE} -> {EXTRACT_DIR} ...")
    with zipfile.ZipFile(ZIP_SOURCE, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Done.")

# Find the data directory (may be nested)
data_dir = EXTRACT_DIR / "data"
if not data_dir.exists():
    # Check if it's directly in the extract root
    data_dir = EXTRACT_DIR / "data_light"
if not data_dir.exists():
    # Search
    for p in EXTRACT_DIR.rglob("dataset.yaml"):
        data_dir = p.parent
        break

print(f"Data directory: {data_dir}")
for split in ["train", "val", "test"]:
    img_dir = data_dir / split / "images"
    if img_dir.exists():
        n = len(list(img_dir.glob("*")))
        print(f"  {split}: {n} images")

with open(data_dir / "dataset.yaml") as f:
    print(f"\ndataset.yaml:\n{f.read()}")

In [ ]:
%pip install -q ultralytics

import ultralytics
ultralytics.checks()

In [ ]:
import yaml

yaml_path = data_dir / "dataset.yaml"
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg["path"] = str(data_dir.resolve())
cfg["train"] = "train/images"
cfg["val"] = "val/images"
cfg["test"] = "test/images"

with open(yaml_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"Updated dataset path -> {cfg['path']}")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=str(data_dir / "dataset.yaml"),
    epochs=100,
    imgsz=640,
    batch=32,
    lr0=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    augment=True,
    device=0,
    workers=4,
    project="/content/grabandgo/runs",
    name="yolo11n_rpc",
    exist_ok=True,
    verbose=True,
)

In [ ]:
metrics = model.val(
    data=str(data_dir / "dataset.yaml"),
    batch=32,
    imgsz=640,
    device=0,
)

print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")

In [ ]:
best = "/content/grabandgo/runs/yolo11n_rpc/weights/best.pt"
model = YOLO(best)
model.export(format="onnx", imgsz=640, half=True)
print("Exported to ONNX")

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

out_dir = Path("/content/grabandgo/output")
out_dir.mkdir(parents=True, exist_ok=True)

for f in Path("/content/grabandgo/runs/yolo11n_rpc/weights").glob("*"):
    shutil.copy2(f, out_dir / f.name)

for f in Path("/content/grabandgo/runs/yolo11n_rpc").glob("*.onnx"):
    shutil.copy2(f, out_dir / f.name)

zip_path = "/content/grabandgo_trained.zip"
shutil.make_archive(zip_path.replace(".zip", ""), "zip", str(out_dir))
print(f"Downloading: {zip_path}")
files.download(zip_path)